# Онлайн-деканат: запуск в Google Colab

Ноутбук ставит локальный Tesseract, скачивает решение, прогоняет архив кейса и
считает метрики `evaluate.py` организаторов. Внешние OCR/LLM API не используются:
всё считается внутри этой машины Colab на CPU.

1. Укажите адрес репозитория в первой ячейке.
2. Запустите все ячейки (`Среда выполнения → Выполнить все`).
3. В ячейке загрузки выберите zip-архив кейса (или укажите путь на Google Drive).

In [ ]:
REPO_URL = "https://github.com/AShevts5/UMIRHack-3"  # репозиторий решения
REPO_DIR = "/content/umir-online-dean"

## 1. Установка (≈ 1–2 минуты)

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-rus > /dev/null
!tesseract --version | head -1 && tesseract --list-langs | tail -n +2 | tr '\n' ' '
!git clone -q --depth 1 "$REPO_URL" "$REPO_DIR" || (cd "$REPO_DIR" && git pull -q)
%cd $REPO_DIR
!pip -q install -r requirements.txt

## 2. Архив кейса

Загрузите zip с `documents.json`, `questions.json` и папкой `documents/`
(для открытого набора — ещё `reference_results.json` и `evaluate.py`).
Если архив уже лежит на Google Drive, впишите путь к распакованной папке в `ARCHIVE`
и пропустите загрузку.

In [ ]:
import glob, os, zipfile
ARCHIVE = ""  # например "/content/drive/MyDrive/online-dean-participants-v1"

if not ARCHIVE:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        with zipfile.ZipFile(name) as zf:
            zf.extractall("/content/archive")
    manifests = glob.glob("/content/archive/**/documents.json", recursive=True)
    ARCHIVE = os.path.dirname(manifests[0])
print("Архив:", ARCHIVE)

## 3. Пакетный прогон → `results.json`

In [ ]:
!python -m backend.scripts.run_batch --documents "$ARCHIVE/documents.json" --questions "$ARCHIVE/questions.json" --root "$ARCHIVE" --output /content/results.json --quiet
print(open("/content/results.json", encoding="utf-8").read()[:600])

## 4. Метрики (если в архиве есть эталон и `evaluate.py`)

In [ ]:
if os.path.exists(f"{ARCHIVE}/reference_results.json"):
    !python "$ARCHIVE/evaluate.py" --reference "$ARCHIVE/reference_results.json" --predictions /content/results.json
else:
    print("Эталона нет (закрытый набор): results.json готов к сдаче.")

## 5. Один документ и свой вопрос

In [ ]:
from backend.app.services.pipeline import Pipeline
import json

pipeline = Pipeline()
entry = json.load(open(f"{ARCHIVE}/documents.json", encoding="utf-8"))[0]
result = pipeline.analyze_path(os.path.join(ARCHIVE, entry["image"]), entry["document_type"])
print("Поля:", json.dumps(result.fields, ensure_ascii=False, indent=2))
print("Проверка:", result.validation_message())
answer = result.single_answer("Кто принимает окончательное решение о принятии документа?")
print("Ответ:", answer.answer, answer.sources)

## 6. Скачать результат

In [ ]:
from google.colab import files
files.download("/content/results.json")